##### 1. Load Environment and Initialize OpenAI Client.

In [33]:
import dotenv
import os


# Ensure OPENAI_API_KEY is set in the environment variables.
dotenv.load_dotenv()  # Load environment variables from .env file if it exists

if "OPENAI_API_KEY" not in os.environ:
    raise EnvironmentError("OPENAI_API_KEY is not set in the environment variables.")
elif not os.environ.get("OPENAI_API_KEY").startswith("sk-"):
    raise EnvironmentError("OPENAI_API_KEY is set but invalid. Please provide a valid API key.")

# Initialize the OpenAI Client.
try:
    import openai
    client = openai.OpenAI()
except Exception as e:
    print("Unable to Initialize OpenAI Client: ", str(e))

##### 2. Fetch the website content for creating webiste brochure.

In [34]:
from httpx import Client
from bs4 import BeautifulSoup

# Fetch the website.
httpx_client = Client(verify=False)
html_content = httpx_client.get('https://felixwealth.in/').text

# Parse uisng BS4
content = BeautifulSoup(html_content)

In [35]:
# Extract the title.
title = content.title.string.strip() if content.title and content.title.string else "No title found"

# Strip elements that don't contribute to the brochure text.
if content.body:
    for tag in content.body(["script", "style", "img", "input", "noscript", "svg"]):
        tag.decompose()
    text = content.body.get_text(separator="\n", strip=True)
else:
    text = ""

# Collect on-site links for follow-up crawling.
from urllib.parse import urljoin, urlparse

base_url = "https://felixwealth.in/"
base_host = urlparse(base_url).netloc
links = []

In [36]:
from IPython.display import display, Markdown
import re

stream = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that summarizes website content into a brochure. Respond with raw markdown only. Do not wrap your response in ``` code fences."},
        {
            "role": "user",
            "content": f"Summarize the following content into a brochure format:\n\nTitle: {title}\n\nText: {text[:2000]} \n\nLinks: {links[:10]}"
        }
    ],
    max_tokens=1000,
    temperature=0.7,
    stream=True
)


def strip_md_fence(s: str) -> str:
    # Remove a leading ```markdown / ``` fence and a trailing ``` if present.
    s = re.sub(r"^\s*```(?:markdown)?\s*\n?", "", s)
    s = re.sub(r"\n?```\s*$", "", s)
    return s


brochure_summary = ""
display_handle = display(Markdown(""), display_id=True)

for event in stream:
    for choice in getattr(event, "choices", []) or []:
        delta_content = getattr(choice.delta, "content", None)
        if delta_content:
            brochure_summary += delta_content
            display_handle.update(Markdown(strip_md_fence(brochure_summary)))

# Felix Wealth - Intelligence Behind Advice

## Experience the Future of Wealth Management

### Advanced Features

- **AI-Powered Portfolio Management**: The most advanced system offering precision, speed, and beauty for modern wealth managers.
- **Real-Time Market Monitoring**: Felix keeps an eye on markets, portfolios, and risk, allowing you to focus on your clients.

### Seamless Workspace

- **Unified Surface**: Manage assets, risk, allocation, and actions all in one view.
- **Portfolio Overview**: Access live data on total assets, gains, and risk profiles at a glance.

### Performance Analytics

- **Detailed Insights**: Track performance over various timeframes with precise analytics.
- **AI Recommendations**: Optimize your portfolio with AI-driven advice, such as increasing allocations in Mid-Cap funds.

### Unmatched Tools

- **Predictive Analytics**: Our AI analyzes over 50,000 data points for accurate market trend forecasts.
- **Risk Radar**: Utilize real-time risk assessments with Monte Carlo simulations.

### Automated Solutions

- **One-Click Reporting**: Generate white-labeled PDF & PPT reports effortlessly.
- **Client Portal**: Offer clients a beautiful, read-only experience.
- **Instant Rebalancing**: Get drift analysis and tap-to-fix rebalancing suggestions.

### Proven Success

- **Managed Assets**: Over ₹50Cr managed.
- **Portfolios Built**: 2,000+ portfolios.
- **Advisors**: Join 500+ advisors.
- **Uptime**: Enjoy 99.9% uptime reliability.

### Get Started Today

- **Free Trial Available**: Experience Felix Wealth's cutting-edge solutions with a free trial.
- **Enterprise Ready**: Scale your practice and join hundreds of satisfied financial advisors.

### Contact Us

- Start your journey to smarter wealth management with Felix Wealth.

---

**Felix Wealth**: The intelligence behind every advice. Always in motion, always in orbit.